In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [24]:
existing_spark = SparkSession.getActiveSession()

if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

Existing Spark session found. Stopping it...


In [3]:
spark = (
        SparkSession.builder
        .appName("Adaptive Query Execution App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d0fa6da6-0b3c-409d-ace0-18f343608306;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [5]:
spark

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "true") # Enable Adaptive Query Execution

26/04/26 10:29:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


### spark.sql.adaptive.coalescePartitions.enabled

In [6]:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

##### When enabled, Spark dynamically reduces the number of shuffle partitions at runtime based on the actual size of data. Normally, spark.sql.shuffle.partitions is fixed (default = 200). If your data is much smaller or larger than expected, the number of partitions may be inefficient (too many tiny files or too few huge partitions).

##### df.repartition(N) is a pre-shuffle hint: it tells Spark how many partitions to use before a shuffle.
##### With AQE + coalescePartitions, Spark reoptimizes the shuffle stage after execution stats are collected.
##### If Spark finds that some partitions are too small, it will coalesce them (merge into fewer partitions).
##### So the final partition count after shuffle may be different from what you requested.

In [7]:
df = spark.range(0, 1000).repartition(50)

In [8]:
result = df.groupBy("id").count()

In [9]:
result.rdd.getNumPartitions()

[Stage 2:==============================================>          (41 + 9) / 50]

1

In [10]:
df.rdd.getNumPartitions()

50

## spark.sql.adaptive.skewJoin.enabled

##### This handles data skew during joins. If one partition is much larger than others, Spark splits that skewed partition into smaller chunks and redistributes them, so no single task becomes a bottleneck.

In [11]:
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

In [12]:
df1 = spark.range(0, 1000).withColumnRenamed("id", "key")
df2 = spark.range(0, 10000).withColumn("key", (col("id") % 2))

In [13]:
joined = df1.join(df2, on="key")

In [14]:
df1.rdd.getNumPartitions()

12

In [15]:
df2.rdd.getNumPartitions()

12

In [16]:
joined.rdd.getNumPartitions()

12

In [17]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [key])
:- Project [id#11L AS key#13L]
:  +- Range (0, 1000, step=1, splits=Some(12))
+- Project [id#15L, (id#15L % cast(2 as bigint)) AS key#17L]
   +- Range (0, 10000, step=1, splits=Some(12))

== Analyzed Logical Plan ==
key: bigint, id: bigint
Project [key#13L, id#15L]
+- Join Inner, (key#13L = key#17L)
   :- Project [id#11L AS key#13L]
   :  +- Range (0, 1000, step=1, splits=Some(12))
   +- Project [id#15L, (id#15L % cast(2 as bigint)) AS key#17L]
      +- Range (0, 10000, step=1, splits=Some(12))

== Optimized Logical Plan ==
Project [key#13L, id#15L]
+- Join Inner, (key#13L = key#17L)
   :- Project [id#11L AS key#13L]
   :  +- Range (0, 1000, step=1, splits=Some(12))
   +- Project [id#15L, (id#15L % 2) AS key#17L]
      +- Filter isnotnull((id#15L % 2))
         +- Range (0, 10000, step=1, splits=Some(12))

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(2) Project [key#13L, id#15L]
   +- *(2) Broadc

## spark.sql.adaptive.join.enabled

##### This lets Spark change the join strategy at runtime (e.g., switch from Sort-Merge Join → Broadcast Hash Join) if actual data sizes are smaller/larger than expected.

In [18]:
spark.conf.set("spark.sql.adaptive.join.enabled", "true")

In [19]:
small_df = spark.range(0, 100).withColumnRenamed("id", "key")

In [20]:
large_df = spark.range(0, 10000000).withColumn("key", (col("id") % 2))

In [21]:
joined = small_df.join(large_df, "key")

In [22]:
joined.explain(False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#24L, id#26L]
   +- BroadcastHashJoin [key#24L], [key#28L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=183]
      :  +- Project [id#22L AS key#24L]
      :     +- Range (0, 100, step=1, splits=12)
      +- Project [id#26L, (id#26L % 2) AS key#28L]
         +- Filter isnotnull((id#26L % 2))
            +- Range (0, 10000000, step=1, splits=12)




In [23]:
joined.explain(mode="cost") # mode = formatted, codegen, cost

== Optimized Logical Plan ==
Project [key#24L, id#26L], Statistics(sizeInBytes=67.1 GiB)
+- Join Inner, (key#24L = key#28L), Statistics(sizeInBytes=89.4 GiB)
   :- Project [id#22L AS key#24L], Statistics(sizeInBytes=800.0 B)
   :  +- Range (0, 100, step=1, splits=Some(12)), Statistics(sizeInBytes=800.0 B, rowCount=100)
   +- Project [id#26L, (id#26L % 2) AS key#28L], Statistics(sizeInBytes=114.4 MiB)
      +- Filter isnotnull((id#26L % 2)), Statistics(sizeInBytes=76.3 MiB)
         +- Range (0, 10000000, step=1, splits=Some(12)), Statistics(sizeInBytes=76.3 MiB, rowCount=1.00E+7)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [key#24L, id#26L]
   +- BroadcastHashJoin [key#24L], [key#28L], Inner, BuildLeft, false
      :- BroadcastExchange HashedRelationBroadcastMode(List(input[0, bigint, false]),false), [plan_id=183]
      :  +- Project [id#22L AS key#24L]
      :     +- Range (0, 100, step=1, splits=12)
      +- Project [id#26L, (id#26L % 2) AS key#28L]
         +